<a href="https://colab.research.google.com/github/bsyeol/26AI_project/blob/main/notebooks/01_data_prep_1104.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01. 데이터 정리 (중간 발표용)

**Sequence-Aware TabPFN-3.5 for Financial Fraud Detection**
AI Hub 이상 판별을 위한 금융거래 정보 및 사용자 패턴 합성데이터 (No. 71925) 중 전자금융공동망 데이터

## 목적
모든 팀원이 **같은 데이터, 같은 분할, 같은 설정**으로 모델을 비교할 수 있도록 공통 파일을 만든다.
이 노트북은 **한 명이 한 번만** 실행한다. 다른 팀원은 결과 파일만 읽어서 사용한다.

## 입력
- 구글 드라이브 `26-2_AI_project/TL_전자금융공동망.zip`
- 그중 CSV 1개: `21-2_전자금융공동망_데이터_2024_1분기.csv`

## 처리 과정
| 셀 | 내용 |
|---|---|
| P1 | zip 안의 CSV 목록 확인 (압축 해제 없이) |
| P1-1 | CSV별 기간, 거래 수, 이상거래 수 비교 → 파일 선택 근거 |
| P2 | 선택한 CSV 읽기, 형 변환, 3시간 bin 번호 생성, 정렬 |
| P3 | 날짜 기준 분할 (train 70 / val 15 / test 15), 같은 날은 같은 구간 |
| P4 | 실험 공통 설정 (seed, 시퀀스 길이, 임베딩 차원 등) |
| P5 | seed별 TabPFN context 행 번호 (정상:이상 = 20:1, 최대 10,000행) |
| P6 | 결과 저장 (덮어쓰기 방지) |

## 출력 (드라이브 `processed/mid/`)
| 파일 | 내용 |
|---|---|
| `mid_v1.csv` | 선택한 데이터 + `split` 컬럼 |
| `split_info.json` | 원본 파일명, 분할 날짜, 구간별 행 수와 이상거래 수, MD5 |
| `settings.json` | 모든 모델이 공통으로 쓰는 실험 설정 |
| `context_seed0~2.npy` | seed별 TabPFN context 행 번호 |

## 결과
| 구간 | 기간 | 거래 수 | 이상거래 |
|---|---|---|---|
| train | 2024-01-01 ~ 03-02 | 229,653 | 1,207 (0.53%) |
| val | 03-03 ~ 03-18 | 49,404 | 386 (0.78%) |
| test | 03-19 ~ 03-31 | 50,264 | 316 (0.63%) |

`mid_v1.csv` MD5: `c7d0fe2cb276cf9d2f2f55834e7827aa`

## 2024년 1분기를 고른 이유
- 2021년 3분기 파일은 9월 한 달치만 있어 이상거래가 83건뿐이었다 (test 18건).
- 2024년 1분기는 이상거래가 1,909건으로 가장 많아 val, test 모두 300건 이상 확보된다.

## 주의
- 이 노트북을 다시 실행하면 분할이 바뀔 수 있다. 다시 만들 때는 팀원에게 먼저 알린다.
- `이상거래유형`, `이상거래설명`은 라벨에서 파생된 컬럼이므로 모델 입력에 쓰지 않는다.
- 원본 데이터는 AI Hub 이용약관에 따라 이 레포에 올리지 않는다.

In [10]:
# =====================================================================
# [P1] zip 안의 CSV 목록 확인
# - 드라이브에서 TL_전자금융공동망.zip을 찾습니다(맥에서 올린 한글 파일명도 처리).
# - 압축을 풀지 않고 zip 안의 CSV 파일 이름만 보여줍니다.
# - 출력된 목록에서 쓸 파일 하나를 골라 P2의 CSV_NAME에 넣습니다.
# =====================================================================
from google.colab import drive
drive.mount('/content/drive')

import os, io, json, hashlib, zipfile, unicodedata
import numpy as np
import pandas as pd

ROOT     = '/content/drive/MyDrive/26-2_AI_project'
MID_DIR  = f'{ROOT}/processed/mid'
ZIP_NAME = 'TL_전자금융공동망.zip'

ZIP_PATH = None
for dirpath, _, filenames in os.walk(ROOT):
    for name in filenames:
        if unicodedata.normalize('NFC', name) == ZIP_NAME:
            ZIP_PATH = os.path.join(dirpath, name)
assert ZIP_PATH, f'{ZIP_NAME}을 찾지 못했습니다'

def clean_name(info):
    name = info.filename
    if not info.flag_bits & 0x800:
        try:
            name = name.encode('cp437').decode('cp949')
        except (UnicodeEncodeError, UnicodeDecodeError):
            pass
    return unicodedata.normalize('NFC', os.path.basename(name))

with zipfile.ZipFile(ZIP_PATH) as z:
    MEMBERS = {clean_name(i): i.filename for i in z.infolist() if i.filename.lower().endswith('.csv')}

print(ZIP_PATH)
for name in sorted(MEMBERS):
    print('   ', name)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/26-2_AI_project/TL_전자금융공동망.zip
    21-2_전자금융공동망_데이터_2021_3분기.csv
    21-2_전자금융공동망_데이터_2021_4분기.csv
    21-2_전자금융공동망_데이터_2022_1분기.csv
    21-2_전자금융공동망_데이터_2022_2분기.csv
    21-2_전자금융공동망_데이터_2022_3분기.csv
    21-2_전자금융공동망_데이터_2022_4분기.csv
    21-2_전자금융공동망_데이터_2023_1분기.csv
    21-2_전자금융공동망_데이터_2023_2분기.csv
    21-2_전자금융공동망_데이터_2023_3분기.csv
    21-2_전자금융공동망_데이터_2023_4분기.csv
    21-2_전자금융공동망_데이터_2024_1분기.csv
    21-2_전자금융공동망_데이터_2024_2분기.csv
    21-2_전자금융공동망_데이터_2024_3분기.csv
    21-2_전자금융공동망_데이터_2024_4분기.csv


In [11]:
# =====================================================================
# [P1-1] zip 안 CSV별 요약
# - 각 CSV에서 거래일자, 이상거래여부 두 컬럼만 읽어 빠르게 비교합니다.
# - 이상거래가 충분한 파일을 고르기 위한 셀입니다.
# =====================================================================
rows = []
with zipfile.ZipFile(ZIP_PATH) as z:
    for name in sorted(MEMBERS):
        d = pd.read_csv(io.BytesIO(z.read(MEMBERS[name])), encoding='utf-8-sig',
                        usecols=['거래일자', '이상거래여부'], dtype=str)
        y = pd.to_numeric(d['이상거래여부'])
        rows.append({'file': name, '시작': d['거래일자'].min(), '끝': d['거래일자'].max(),
                     '거래수': len(d), '이상거래': int(y.sum())})
print(pd.DataFrame(rows).to_string(index=False))

                         file       시작        끝    거래수  이상거래
21-2_전자금융공동망_데이터_2021_3분기.csv 20210901 20210930  82050    83
21-2_전자금융공동망_데이터_2021_4분기.csv 20211001 20211231 274136   546
21-2_전자금융공동망_데이터_2022_1분기.csv 20220101 20220331 342558   910
21-2_전자금융공동망_데이터_2022_2분기.csv 20220401 20220630 256532   995
21-2_전자금융공동망_데이터_2022_3분기.csv 20220701 20220930 292259  1236
21-2_전자금융공동망_데이터_2022_4분기.csv 20221001 20221231 260228  1011
21-2_전자금융공동망_데이터_2023_1분기.csv 20230101 20230331 274648   851
21-2_전자금융공동망_데이터_2023_2분기.csv 20230401 20230630 298749   840
21-2_전자금융공동망_데이터_2023_3분기.csv 20230701 20230930 301285  1254
21-2_전자금융공동망_데이터_2023_4분기.csv 20231001 20231231 286663  1630
21-2_전자금융공동망_데이터_2024_1분기.csv 20240101 20240331 329321  1909
21-2_전자금융공동망_데이터_2024_2분기.csv 20240401 20240630 356291  1703
21-2_전자금융공동망_데이터_2024_3분기.csv 20240701 20240930 324442  1046
21-2_전자금융공동망_데이터_2024_4분기.csv 20241001 20241231 262266  1253


In [12]:
# =====================================================================
# [P2] 고른 CSV 하나 읽기와 정리
# - CSV_NAME: P1 목록에서 고른 파일 이름
# - zip에서 그 파일 하나만 바로 읽습니다(압축 해제 없음).
# - 모든 값을 문자로 읽은 뒤 거래금액, 거래시간대, 이상거래여부만 숫자로 바꿉니다.
# - date: 거래 날짜 / bin_of_day: 하루 중 3시간 구간 번호(0~7)
#   bin_id: 첫 날부터 센 전체 구간 번호(시퀀스 모델용)
# - 날짜, 시간대, 계좌 순으로 정렬해 누가 돌려도 행 순서가 같게 합니다.
# =====================================================================
CSV_NAME = '21-2_전자금융공동망_데이터_2024_1분기.csv'
USER  = '출금계좌일련번호'
LABEL = '이상거래여부'

with zipfile.ZipFile(ZIP_PATH) as z:
    raw = z.read(MEMBERS[CSV_NAME])

try:
    mid = pd.read_csv(io.BytesIO(raw), encoding='utf-8-sig', dtype=str)
except UnicodeDecodeError:
    mid = pd.read_csv(io.BytesIO(raw), encoding='cp949', dtype=str)

for c in ['거래금액', '거래시간대', LABEL]:
    mid[c] = pd.to_numeric(mid[c], errors='coerce')

mid['date']       = pd.to_datetime(mid['거래일자'], format='%Y%m%d', errors='coerce')
mid['bin_of_day'] = mid['거래시간대'] // 3
mid['bin_id']     = (mid['date'] - mid['date'].min()).dt.days * 8 + mid['bin_of_day']

mid = mid.sort_values(['date', 'bin_of_day', USER], kind='stable').reset_index(drop=True)

print(mid.shape, mid['date'].min().date(), '~', mid['date'].max().date())
print('변환 실패:', mid[['date', '거래시간대', '거래금액', LABEL]].isna().sum().to_dict())
print('시간대 값:', sorted(mid['거래시간대'].dropna().unique()))
print(f'이상거래: {int(mid[LABEL].sum())} / {len(mid)} ({mid[LABEL].mean():.3%})')
print('출금계좌:', mid[USER].nunique())

(329321, 15) 2024-01-01 ~ 2024-03-31
변환 실패: {'date': 0, '거래시간대': 0, '거래금액': 0, '이상거래여부': 0}
시간대 값: [np.int64(0), np.int64(3), np.int64(6), np.int64(9), np.int64(12), np.int64(15), np.int64(18), np.int64(21)]
이상거래: 1909 / 329321 (0.580%)
출금계좌: 27694


In [13]:
#P3
RATIO = (0.70, 0.15, 0.15)

daily = mid.groupby('date').size().cumsum() / len(mid)
TRAIN_END = daily[daily <= RATIO[0]].index.max()
VAL_END   = daily[daily <= RATIO[0] + RATIO[1]].index.max()

mid['split'] = 'test'
mid.loc[mid['date'] <= VAL_END,   'split'] = 'val'
mid.loc[mid['date'] <= TRAIN_END, 'split'] = 'train'

print(mid.groupby('split').agg(
    시작=('date', 'min'), 끝=('date', 'max'),
    거래수=(LABEL, 'size'), 이상거래=(LABEL, 'sum'), 비율=(LABEL, 'mean'),
).reindex(['train', 'val', 'test']))

              시작          끝     거래수  이상거래        비율
split                                              
train 2024-01-01 2024-03-02  229653  1207  0.005256
val   2024-03-03 2024-03-18   49404   386  0.007813
test  2024-03-19 2024-03-31   50264   316  0.006287


In [14]:
#P4
SETTINGS = {
    'SEEDS': [0, 1, 2],
    'CONTEXT_SIZE': 10000,
    'NORMAL_PER_FRAUD': 20,
    'PRED_BATCH': 20000,
    'K': 16,
    'H_DIM': 32,
    'FLIP_R': 4,
    'T2V_K': 8,
    'EPOCHS': 20,
    'PATIENCE': 3,
    'BATCH': 1024,
    'LR': 1e-3,
    'PARAM_BUDGET': [38000, 48000],
    'BASE_FEATS': ['log_amount', '거래시간대', 'dow', '매체구분', '자금구분',
                   '출금금융회사일련번호', '입금금융회사일련번호'],
    'DROP_COLS': ['이상거래유형', '이상거래설명'],
    'LABEL': LABEL,
    'USER': USER,
}

In [15]:
#P5
def make_context(seed):
    rng = np.random.default_rng(seed)
    tr = mid[mid['split'] == 'train']
    fraud  = tr.index[tr[LABEL] == 1].to_numpy()
    normal = tr.index[tr[LABEL] == 0].to_numpy()
    n_f = min(len(fraud), SETTINGS['CONTEXT_SIZE'] // (SETTINGS['NORMAL_PER_FRAUD'] + 1))
    n_n = min(len(normal), n_f * SETTINGS['NORMAL_PER_FRAUD'])
    return np.sort(np.concatenate([rng.choice(fraud, n_f, replace=False),
                                   rng.choice(normal, n_n, replace=False)]))

CONTEXT = {s: make_context(s) for s in SETTINGS['SEEDS']}
for s, idx in CONTEXT.items():
    print(f'seed {s}: {len(idx)}행, 이상거래 {int(mid.loc[idx, LABEL].sum())}건')

seed 0: 9996행, 이상거래 476건
seed 1: 9996행, 이상거래 476건
seed 2: 9996행, 이상거래 476건


In [18]:
# =====================================================================
# [P5-1] 기존 mid 폴더 삭제 (다시 만들 때만 실행)
# - 이전에 저장한 2021년 3분기 결과를 지웁니다.
# - 팀원이 이미 그 파일로 실험을 시작했다면, 지우기 전에 먼저 알려야 합니다.
# =====================================================================
#import shutil
#shutil.rmtree(MID_DIR, ignore_errors=True)
#print('삭제 후 존재 여부:', os.path.exists(MID_DIR))

삭제 후 존재 여부: False


In [19]:

#P6
assert not os.path.exists(f'{MID_DIR}/split_info.json'), '이미 저장된 파일이 있습니다. 덮어쓰지 마세요.'
os.makedirs(MID_DIR, exist_ok=True)

csv_path = f'{MID_DIR}/mid_v1.csv'
mid.to_csv(csv_path, index=False, encoding='utf-8-sig')
md5 = hashlib.md5(open(csv_path, 'rb').read()).hexdigest()

info = {
    'source_zip': ZIP_NAME,
    'source_file': CSV_NAME,
    'train_end':   str(TRAIN_END.date()),
    'val_end':     str(VAL_END.date()),
    'rows':        {k: int(v) for k, v in mid['split'].value_counts().items()},
    'fraud':       {k: int(v) for k, v in mid.groupby('split')[LABEL].sum().items()},
    'md5':         md5,
}
json.dump(info, open(f'{MID_DIR}/split_info.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
json.dump(SETTINGS, open(f'{MID_DIR}/settings.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
for s, idx in CONTEXT.items():
    np.save(f'{MID_DIR}/context_seed{s}.npy', idx)

print(os.listdir(MID_DIR))
print(json.dumps(info, ensure_ascii=False, indent=2))

['mid_v1.csv', 'split_info.json', 'settings.json', 'context_seed0.npy', 'context_seed1.npy', 'context_seed2.npy']
{
  "source_zip": "TL_전자금융공동망.zip",
  "source_file": "21-2_전자금융공동망_데이터_2024_1분기.csv",
  "train_end": "2024-03-02",
  "val_end": "2024-03-18",
  "rows": {
    "train": 229653,
    "test": 50264,
    "val": 49404
  },
  "fraud": {
    "test": 316,
    "train": 1207,
    "val": 386
  },
  "md5": "c7d0fe2cb276cf9d2f2f55834e7827aa"
}
